# M02-02 — Schema y tipos

[← Anterior](02-lab-ingesta-csv-json.ipynb) · [Siguiente →](04-lab-calidad-limpieza.ipynb)

Este fichero es el **guion**. No lo rellenes aquí: **crea tu propio notebook** y ve construyéndolo celda a celda.

## Qué vas a hacer

Leer pedidos, líneas y eventos con schema (o cast), nombres en snake_case e importes/fechas casteados.

## 0 — Crea tu notebook

1. En el explorador, abre la carpeta `notebooks/trabajo/`.
2. Clic derecho → **New File…**
3. Nombre exacto: `M02-02-schema-tipos.ipynb` (incluye `.ipynb`).
4. Ábrelo. Arriba a la derecha (o `F1` → `Notebook: Select Notebook Kernel`) elige **Python (NovaShop)**.
5. Deja **este** guion a un lado (pestaña) y escribe **solo** en el tuyo.

## Cómo organizar *tu* notebook (siempre)

En cada paso creas **dos celdas**, en este orden:

1. **Markdown** — qué vas a hacer y por qué, con tus palabras.
2. **Código** — el de la celda de código del paso. Lo ejecutas (`Shift+Enter`), miras la salida y, si no cuadra, lo mejoras.

No dejes un muro de código sin explicación. Un notebook se lee de arriba abajo, como un cuaderno.

> Kernel **Python (NovaShop)**. Si no aparece: terminal → `bash .devcontainer/setup.sh` → vuelve a elegir kernel.


### Paso 1 — Arranque

En *tu* notebook: una celda Markdown que explique esto (con tus palabras):

Celda 0 y sesión. Este lab parte de raw, no de lo que tenías en memoria ayer.

Debajo, una celda de código. El código está **en la celda siguiente** (márcalo y llévatelo). Ejecuta (`Shift+Enter`).

**Comprueba.** Sesión lista. `RAW` True.

**Por qué este paso.** Cada notebook es autónomo: no asumas variables de otro fichero.


In [ ]:
import sys
from pathlib import Path

# El notebook puede estar en trabajo/; subimos hasta encontrar el repo.
_here = Path.cwd().resolve()
ROOT = next(
    p
    for p in [_here, *_here.parents]
    if (p / "labs" / "_shared" / "session.py").is_file()
)
sys.path.insert(0, str(ROOT / "labs" / "_shared"))

from paths import RAW, STAGING, CURATED  # rutas absolutas, no Path("data/raw")
from session import get_spark

print("ROOT   ", ROOT)
print("RAW    ", RAW, "existe:", RAW.is_dir())
print("STAGING", STAGING)
print("CURATED", CURATED)


spark = get_spark("novashop-m02")


### Paso 2 — Schema de pedidos y snake_case

En *tu* notebook: una celda Markdown que explique esto (con tus palabras):

El fichero trae camelCase. El pipeline interno habla snake_case. Declaro el schema y renombro.

Debajo, una celda de código. El código está **en la celda siguiente** (márcalo y llévatelo). Ejecuta (`Shift+Enter`).

**Comprueba.** Cinco columnas ya renombradas. `order_ts_raw` sigue string. Count **800**.

**Por qué este paso.** Tipar no borra filas. Las fechas raras se arreglan en el siguiente paso.


In [ ]:
from pyspark.sql.types import StructType, StructField, StringType

orders_raw_schema = StructType([
    StructField("OrderId", StringType(), True),
    StructField("CustomerId", StringType(), True),
    StructField("OrderDate", StringType(), True),
    StructField("Status", StringType(), True),
    StructField("Channel", StringType(), True),
])
orders = (
    spark.read.option("header", True)
    .schema(orders_raw_schema)
    .csv(str(RAW / "orders.csv"))
    .withColumnRenamed("OrderId", "order_id")
    .withColumnRenamed("CustomerId", "customer_id")
    .withColumnRenamed("OrderDate", "order_ts_raw")
    .withColumnRenamed("Status", "status")
    .withColumnRenamed("Channel", "channel")
)
orders.printSchema()
print(orders.count())


### Paso 3 — Timestamp con dos formatos

En *tu* notebook: una celda Markdown que explique esto (con tus palabras):

Hay ISO y dd/MM/yyyy. Un solo to_timestamp deja nulos. coalesce de dos formatos las recupera.

Debajo, una celda de código. El código está **en la celda siguiente** (márcalo y llévatelo). Ejecuta (`Shift+Enter`).

**Comprueba.** `0` nulos en `order_ts`. Tipo `timestamp`.

**Por qué este paso.** Si solo usas ISO, las 3 filas sucias mueren como nulo.


In [ ]:
from pyspark.sql.functions import col, coalesce, to_timestamp

orders = orders.withColumn(
    "order_ts",
    coalesce(
        to_timestamp(col("order_ts_raw"), "yyyy-MM-dd HH:mm:ss"),
        to_timestamp(col("order_ts_raw"), "dd/MM/yyyy"),
    ),
).drop("order_ts_raw")
print("nulos de fecha", orders.where(col("order_ts").isNull()).count())
orders.printSchema()


### Paso 4 — Líneas: enteros y decimales

En *tu* notebook: una celda Markdown que explique esto (con tus palabras):

En el CSV unit_price es texto. DecimalType es el tipo de dinero del curso. Reasigno items = items.withColumn(...).

Debajo, una celda de código. El código está **en la celda siguiente** (márcalo y llévatelo). Ejecuta (`Shift+Enter`).

**Comprueba.** `qty` integer, `unit_price`/`discount` decimal. `show` ya no pone comillas.

**Por qué este paso.** Si no reasignas, `unit_price` sigue string en el objeto viejo.


In [ ]:
from pyspark.sql.types import IntegerType, DecimalType

items = (
    spark.read.option("header", True).csv(str(RAW / "order_items.csv"))
    .withColumn("qty", col("qty").cast(IntegerType()))
    .withColumn("unit_price", col("unit_price").cast(DecimalType(10, 2)))
    .withColumn("discount", col("discount").cast(DecimalType(5, 2)))
)
items.printSchema()
items.select("unit_price").limit(3).show()


### Paso 5 — Eventos con schema

En *tu* notebook: una celda Markdown que explique esto (con tus palabras):

JSONL infiere bien casi siempre; el schema evita que ts se quede string el día que llegue un fichero raro.

Debajo, una celda de código. El código está **en la celda siguiente** (márcalo y llévatelo). Ejecuta (`Shift+Enter`).

**Comprueba.** 2500 filas; `ts` en `timestamp`.

**Por qué este paso.** Declarar el contrato es más barato que depurar un inferido distinto mañana.


In [ ]:
from pyspark.sql.types import TimestampType

events_schema = StructType([
    StructField("event_id", StringType(), False),
    StructField("customer_id", StringType(), True),
    StructField("event_type", StringType(), True),
    StructField("ts", TimestampType(), True),
    StructField("session_id", StringType(), True),
    StructField("page", StringType(), True),
    StructField("product_id", StringType(), True),
])
events = spark.read.schema(events_schema).json(str(RAW / "events.jsonl"))
events.printSchema()
print(events.count())


## Comprueba

Antes de dar el lab por cerrado, vuelve a ejecutar de arriba abajo (**Run All**) y verifica:

`printSchema()` de `orders` (tras el paso 3) e `items` (paso 4):
`orders.order_ts` timestamp; `items.unit_price` `decimal(10,2)`; `items.qty` int.


## Mejora — Catálogo en snake_case y decimal

Lee `products.json` (multiLine), renombra `productId` → `product_id`, `listPrice` → `list_price` y castea `list_price` a `DecimalType(10,2)`. Tres `list_price` nulos es correcto (se limpian en el siguiente lab).

Si te atasca, el código está en la celda siguiente.


In [ ]:
products = (
    spark.read.option("multiLine", True).json(str(RAW / "products.json"))
    .withColumnRenamed("productId", "product_id")
    .withColumnRenamed("listPrice", "list_price")
    .withColumn("list_price", col("list_price").cast(DecimalType(10, 2)))
)
products.printSchema()


## Si algo falla

| Qué ves | Suele ser | Qué haces |
|---------|-----------|-----------|
| Casi todas las fechas nulas | Un solo to_timestamp ISO | Añade `dd/MM/yyyy` en el coalesce |
| unit_price sigue string | No reasignaste | `items = items.withColumn(...)` |
| cannot resolve OrderId | Renombraste y filtraste el nombre viejo | Usa `order_id` a partir de aquí |


## Siguiente

Cuando hayas **comprobado** y (si quieres) **mejorado**, abre [M02-03 calidad](04-lab-calidad-limpieza.ipynb).
